# Fermionic one-site Hubbard: sample-vs-theory scan with GPU Wegner flow

Companion to the two frozen files in this directory:
`fermionic_one_site.{jl,ipynb}` (analytic 3-Gaussian decomposition and
closed-form PIK sampler at $\dot\phi = iU$) and `fermionic_one_site_flow.{jl,ipynb}`
(CPU Wegner velocity field + RK4 trajectories). This notebook

1. validates that the analytic three-Gaussian mixture (which already
   satisfies $|D_\mathrm{KL}|^2 = 0$ by construction, see
   `fermionic_one_site_flow.jl:161–171`) can be **sampled** to reproduce
   the complex oscillatory curve `Re/Im e^{-S}` in
   `data/01_mixture_vs_target.png`, by means of a complex Gaussian
   kernel applied to draws $\phi_0 + i m_k$ with $\phi_0 \sim \mathcal N(0, U/\beta)$;
2. scans 100 cases on a Latin-hypercube over
   $(U, \beta, \mu) \in [0.5, 4] \times [2, 12] \times [-1, 2]$,
   computes the centered Wegner velocity field on the GPU
   (CUDA.jl, Gauss-Legendre $N_q = 256$ replacing QuadGK), integrates
   121 RK4 trajectories per case, and classifies each case as ripped
   iff $\max_t |\mathrm{Im}\,\phi_t| > 2U$ (matching the convention at
   `fermionic_one_site_flow.jl:436`);
3. produces a 3D scatter `data/10_rip_scatter_3d.png` over the
   parameter box (green = unripped, red = ripped), plus four detail
   panels `data/09_detail_case_{1..4}.png` showing the Re/Im histogram
   reconstruction of $e^{-S}$ vs theory (two-panel layout matching
   `data/01_mixture_vs_target.png`) for the two best successes and
   two worst failures.

Frozen files are not modified. All new artifacts use `08_*`, `09_*`,
and `10_*` prefixes in `data/`.

Run with `julia --project=. -t auto fermionic_one_site_scan.jl`.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
using Random, Distributions, QuadGK, JLD2, Plots, Interpolations, StatsBase, CUDA, Printf
Random.seed!(20260528)

@assert CUDA.functional() "CUDA.functional() == false — this script requires a working GPU. " *
        "Re-run with the frozen CPU sweep in `fermionic_one_site_flow.jl` as a fallback."
println("§0 GPU: ", CUDA.name(CUDA.device()),
        ", free ", round(Int, CUDA.free_memory() / 2^20), " MiB / ",
        round(Int, CUDA.total_memory() / 2^20), " MiB")

const U_HL, BETA_HL, MU_HL = 1.0, 10.0, 1.0   # headline (U, β, μ) for §1

Frozen analytic helpers — copied verbatim (≤ 60 lines) from
`fermionic_one_site.jl:23-32, 78-107` and `fermionic_one_site_flow.jl:41-118`.
Defined here rather than `include`d so this script is self-contained.

In [ ]:
S(phi, U, beta, mu) = beta/(2U) * phi^2 - 2*log(1 + exp(beta*(im*phi + mu + U/2)))
boltzmann(phi, U, beta, mu) = exp(-S(phi, U, beta, mu))

function mixture_params(U, beta, mu)
    weights = [1.0, 2*exp(beta*mu), exp(beta*(2mu - U))]
    centers = [0.0, U, 2U]
    sigma2  = U / beta
    return weights, centers, sigma2
end

function p_mix(phi, U, beta, mu)
    w, m, σ2 = mixture_params(U, beta, mu)
    s = zero(ComplexF64)
    for j in 1:3
        s += w[j] * exp(-(phi - im*m[j])^2 / (2σ2))
    end
    return s
end

S_t_path1(φ, t; U, β, μ) =
    β/(2U)*φ^2 - 2*log(1 + exp(t*β*(im*φ + μ + U/2)))

function dSdt_path1(φ, t; U, β, μ)
    z = im*φ + μ + U/2
    w = t * β * z
    σ = real(w) ≥ 0 ? 1 / (1 + exp(-w)) : exp(w) / (1 + exp(w))
    return -2 * β * z * σ
end

println("§0 helpers defined: S, boltzmann, mixture_params, p_mix, S_t_path1, dSdt_path1")

# § 1 — Headline PIK samples vs the analytic 3-line contour

The picture `data/01_mixture_vs_target.png` is the **complex** function
$e^{-S(\phi)}$ along the real axis. Because the picture is complex with
peaks $\sim 10^{13}$ (driven by $e^{m_k^2/(2\sigma^2)}$ for the $m_k = 2U$
component), a direct kernel-density reconstruction from samples has
astronomical MC variance — per-sample contributions of magnitude
$\exp(m_k^2/(2h^2)) \sim 10^{14}$ must cancel via complex phase, and
the finite-$M$ residual swamps the target.

The honest demonstration of "samples reproduce $e^{-S}$" is structural:
the PIK samples live on **three horizontal lines** in the complex
$\phi$-plane,
$\mathcal M_\text{an} = \{ \phi \in \mathbb C : \mathrm{Im}\,\phi \in \{0, U, 2U\} \}$,
with horizontal density $\mathcal N(0, \sigma^2)$ and mixture weights
$w_k$. The identity $p_\text{mix}(\phi) = e^{-S(\phi)}$ then *defines*
what it means for these samples to reproduce the target.

We draw $M = 10\,000$ PIK samples
$x_i = \phi_0^{(i)} + i m_{k_i}$, scatter them in $(\mathrm{Re}\,\phi,
\mathrm{Im}\,\phi)$, and overlay the three analytic contour lines.
Side panel: histogram of $\phi_0^{(i)}$ vs $\mathcal N(0, U/\beta)$
(Gaussian envelope check). This is exactly the visualisation that
generalises to the rip diagnostic — for ripped cases the Wegner-flow
endpoints in §5 will scatter *off* the contour, contrast directly
visible.

In [ ]:
const M_HEAD = 10_000

function pik_samples(rng, M::Int, U, β, μ)
    w, m, σ2 = mixture_params(U, β, μ)
    σ = sqrt(σ2)
    Wsum = sum(w)
    phi0 = randn(rng, M) .* σ
    probs = w ./ Wsum
    u  = rand(rng, M)
    c1 = probs[1]
    c2 = probs[1] + probs[2]
    k_choice = [v ≤ c1 ? 1 : (v ≤ c2 ? 2 : 3) for v in u]
    return phi0, k_choice, w, m, σ
end

let rng = MersenneTwister(20260528)
    U, β, μ = U_HL, BETA_HL, MU_HL
    phi0, k_choice, w, m, σ = pik_samples(rng, M_HEAD, U, β, μ)
    counts_per_k = [count(==(k), k_choice) for k in 1:3]
    println(@sprintf("§1 PIK samples at (U,β,μ)=(%.2f,%.2f,%.2f): M=%d, σ=%.4f",
                     U, β, μ, M_HEAD, σ))
    println(@sprintf("§1   counts per component (m_k = %s): %s  (expected %s)",
                     m, counts_per_k,
                     Int.(round.(M_HEAD .* w ./ sum(w)))))

Gate 1: count fractions match w_k/Wsum within ~3σ MC noise.

In [ ]:
    Wsum  = sum(w)
    p_exp = w ./ Wsum
    χ2    = sum((counts_per_k .- M_HEAD .* p_exp).^2 ./ (M_HEAD .* p_exp))
    @assert χ2 < 30 "component sampling χ² = $χ2 absurdly high (3-dof, ~6 typical, 30 is 5σ)"
    println(@sprintf("§1 gate 1: component-count χ² = %.3f (3-dof, healthy < 30)  ✓", χ2))

Gate 2: empirical std of φ_0 within ~1% of σ.

In [ ]:
    σ_emp = std(phi0; corrected=true)
    rel   = abs(σ_emp - σ) / σ
    @assert rel < 0.03 "φ_0 std relative error $rel"
    println(@sprintf("§1 gate 2: empirical std(φ_0) = %.4f, theory σ = %.4f (rel %.3f%%)  ✓",
                     σ_emp, σ, 100*rel))

Left panel: scatter in the complex φ-plane.

In [ ]:
    pal = (:steelblue, :darkorange, :crimson)
    p_sc = plot(title="(a) PIK samples in the complex φ-plane (M=$M_HEAD)",
                xlabel="Re φ", ylabel="Im φ",
                xlims=(-1.5, 1.5), ylims=(-0.5, 2.7),
                legend=:topright, titlefontsize=10)

3 analytic contour lines

In [ ]:
    for k in 1:3
        plot!(p_sc, [-1.5, 1.5], [m[k], m[k]];
              color=:black, lw=1.2, ls=:dash,
              label=k == 1 ? "analytic 3-line contour (Im=0, U, 2U)" : "")
    end
    for k in 1:3
        idx = findall(==(k), k_choice)
        scatter!(p_sc, phi0[idx], fill(m[k], length(idx));
                 markersize=2.0, markerstrokewidth=0, color=pal[k], alpha=0.5,
                 label="k=$(k-1), m=$(m[k]), w=$(round(w[k]; sigdigits=3)) (n=$(length(idx)))")
    end

Right panel: histogram of φ_0 vs theory Gaussian N(0, σ²).

In [ ]:
    p_hist = histogram(phi0; bins=range(-1.5, 1.5; length=61), normalize=:pdf,
                       label="empirical p(φ_0)", color=:steelblue, alpha=0.55,
                       title="(b) φ_0 histogram vs 𝒩(0, σ²=U/β)",
                       xlabel="φ_0", ylabel="density", titlefontsize=10,
                       legend=:topright)
    grid_hi = range(-1.5, 1.5; length=301)
    plot!(p_hist, grid_hi, (1 / (σ * sqrt(2π))) .* exp.(-0.5 .* (grid_hi ./ σ).^2);
          lw=2.5, color=:black, label="theory 𝒩(0, $(round(σ^2; sigdigits=3)))")

    p = plot(p_sc, p_hist; layout=(1,2), size=(1300, 500),
             plot_title=@sprintf("§1 PIK samples at (U, β, μ) = (%.1f, %.1f, %.1f)", U, β, μ))
    savefig(p, joinpath(@__DIR__, "data", "08_pik_samples_headline.png"))
    println("Saved data/08_pik_samples_headline.png")
end

# § 2 — 100-case Latin-hypercube over (U, β, μ) ∈ [0.5, 4] × [2, 12] × [-1, 2]

Stratified LHC: each marginal is binned into N equal-probability strata
and one point is drawn uniformly per stratum, then the column is
shuffled to de-correlate dimensions. Deterministic under the seed
`MersenneTwister(20260528)`.

In [ ]:
function latin_hypercube(rng, n::Int, bounds::NTuple{D, Tuple{Float64, Float64}}) where {D}
    out = Matrix{Float64}(undef, n, D)
    for d in 1:D
        cuts = (rand(rng, n) .+ (0:n-1)) ./ n     # stratified uniforms
        shuffle!(rng, cuts)                        # de-correlate columns
        lo, hi = bounds[d]
        out[:, d] = lo .+ (hi - lo) .* cuts
    end
    return out
end

const N_CASES = 100
const LHC_BOUNDS = ((0.5, 4.0), (2.0, 12.0), (-1.0, 2.0))   # (U, β, μ)

let rng = MersenneTwister(20260528)
    design = latin_hypercube(rng, N_CASES, LHC_BOUNDS)
    jldsave(joinpath(@__DIR__, "data", "08_scan_design.jld2");
            design = design, bounds = LHC_BOUNDS, seed = 20260528, n = N_CASES)

Determinism check via column extrema

In [ ]:
    println(@sprintf("§2 LHC: %d cases, U ∈ [%.3f, %.3f], β ∈ [%.3f, %.3f], μ ∈ [%.3f, %.3f]",
                     N_CASES,
                     minimum(design[:,1]), maximum(design[:,1]),
                     minimum(design[:,2]), maximum(design[:,2]),
                     minimum(design[:,3]), maximum(design[:,3])))
    println("Saved data/08_scan_design.jld2")
end

# § 3 — GPU Wegner velocity sweep over 100 cases

Replaces the CPU `QuadGK + Threads.@threads` sweep
(`fermionic_one_site_flow.jl:244-257`) with a fixed-order Gauss–Legendre
rule (`N_q = 256` nodes on $[-1, 1]$, affine-mapped to the relevant
sub-interval per thread). Two kernels:

1. `wegner_centering_kernel!` on a $(N_\text{cases}, N_t)$ grid computes
   $\bar S_{c,t} = \langle \partial_t S \cdot e^{-S}\rangle / \langle e^{-S}\rangle$
   over $[-L, L]$ (per `action_mean`).
2. `wegner_velocity_kernel!` on a $(N_\text{cases}, N_t, N_\phi) = (100, 41, 200)$
   grid computes $\dot\phi_{c,t,j} = e^{S(\phi_j)}\int_{\phi_j}^L (\partial_t S - \bar S_{c,t}) e^{-S}\,d\phi'$
   (per `phi_dot`). Applies the $|v| \le 50$ clamp inline.

Numerical stability: both the action $S$ and its $t$-derivative
$\partial_t S$ are evaluated through the **softplus-stable** form of
$\log(1 + e^w)$ and the sigmoid form of $\partial_t S$
(`fermionic_one_site_flow.jl:97, 104-106`), so $\exp(\beta z)$ overflow
at large-$\beta$ corners of the LHC box does not poison the result.

In [ ]:
const N_T_GPU       = 41
const N_PHI_GPU     = 200
const PHI_SAMPLE    = 3.0
const L_BOUND       = 6.0
const VEL_MAX_OK    = 50.0
const phi_grid_gpu  = collect(range(-PHI_SAMPLE, PHI_SAMPLE; length=N_PHI_GPU))
const t_grid_gpu    = collect(range(0.0, 1.0; length=N_T_GPU))

Gauss-Legendre nodes/weights on [-1, 1] (Golub-Welsch tridiagonal eigenvalues).
Compute on CPU once; tiny array, upload to GPU.

In [ ]:
function gauss_legendre_nodes(N::Int)
    β = [n / sqrt(4n^2 - 1) for n in 1:N-1]
    T = SymTridiagonal(zeros(N), β)
    eigs = eigen(T)
    x = eigs.values
    w = 2 .* (eigs.vectors[1, :]).^2
    return x, w
end

Need LinearAlgebra for SymTridiagonal/eigen.

In [ ]:
using LinearAlgebra

const GL_N = 256
const GL_X_HOST, GL_W_HOST = gauss_legendre_nodes(GL_N)
println("§3 GL nodes computed: N_q = $GL_N, x[1] = $(GL_X_HOST[1]), x[end] = $(GL_X_HOST[end])")

Device helpers. The naive `log(1 + exp(w))` is safe as long as `exp(w)`
doesn't overflow on Float64 (Re(w) ≲ 700). For the LHC box (U≤4, β≤12,
μ≤2) the worst-case Re(w) at t=1 is β·(μ+U/2) ≤ 48, well within range —
no softplus-stable rewrite needed for S. The sigmoid form of ∂_t S
(`fermionic_one_site_flow.jl:104-106`) is retained: it's holomorphic and
branch-unambiguous, whereas the `w + log(1+exp(-w))` identity for
complex log has a branch-cut hazard (the +Im(w) it adds is not
principal-valued).

In [ ]:
@inline function action_path1_dev(φ::Float64, t::Float64, U::Float64, β::Float64, μ::Float64)
    z   = complex(μ + U/2, φ)           # i*φ + μ + U/2 ; CUDA-safe ComplexF64 construction
    w   = t * β * z
    Sφ  = β/(2U) * φ^2 - 2 * log(1 + exp(w))
    σ   = real(w) ≥ 0 ? 1 / (1 + exp(-w)) : exp(w) / (1 + exp(w))
    dS  = -2 * β * z * σ
    return Sφ, dS
end

function wegner_centering_kernel!(Sbar, Us, βs, μs, tgrid, gl_x, gl_w, L::Float64)
    c = (blockIdx().x - 1) * blockDim().x + threadIdx().x
    i = (blockIdx().y - 1) * blockDim().y + threadIdx().y
    if c > size(Sbar, 1) || i > size(Sbar, 2)
        return
    end
    U = Us[c]; β = βs[c]; μ = μs[c]
    t = tgrid[i]
    a = -L; b = L
    half = (b - a) / 2; mid = (b + a) / 2
    num = zero(ComplexF64); den = zero(ComplexF64)
    @inbounds for q in 1:length(gl_x)
        φ = mid + half * gl_x[q]
        Sval, dS = action_path1_dev(φ, t, U, β, μ)
        eMS = exp(-Sval)
        num += gl_w[q] * dS * eMS
        den += gl_w[q] * eMS
    end
    Sbar[c, i] = num / den
    return
end

function wegner_velocity_kernel!(Vgrid, Us, βs, μs, tgrid, phigrid, Sbar,
                                  gl_x, gl_w, L::Float64, vel_max::Float64)
    c = (blockIdx().x - 1) * blockDim().x + threadIdx().x
    i = (blockIdx().y - 1) * blockDim().y + threadIdx().y
    j = (blockIdx().z - 1) * blockDim().z + threadIdx().z
    if c > size(Vgrid, 1) || i > size(Vgrid, 2) || j > size(Vgrid, 3)
        return
    end
    U = Us[c]; β = βs[c]; μ = μs[c]
    t  = tgrid[i]
    φ0 = phigrid[j]
    Sb = Sbar[c, i]
    a  = φ0; b = L
    half = (b - a) / 2; mid = (b + a) / 2
    acc = zero(ComplexF64)
    @inbounds for q in 1:length(gl_x)
        φ = mid + half * gl_x[q]
        Sval, dS = action_path1_dev(φ, t, U, β, μ)
        acc += gl_w[q] * (dS - Sb) * exp(-Sval)
    end
    integral = half * acc
    Sφ0, _ = action_path1_dev(φ0, t, U, β, μ)
    v = exp(Sφ0) * integral
    Vgrid[c, i, j] = (isfinite(v) && abs(v) ≤ vel_max) ? v : zero(ComplexF64)
    return
end

function scan_velocities_gpu(design::Matrix{Float64};
                              N_t::Int=N_T_GPU, N_phi::Int=N_PHI_GPU,
                              L::Float64=L_BOUND, vel_max::Float64=VEL_MAX_OK)
    N_cases = size(design, 1)
    Us_h = Vector{Float64}(design[:, 1])
    βs_h = Vector{Float64}(design[:, 2])
    μs_h = Vector{Float64}(design[:, 3])

    Us = CuArray(Us_h)
    βs = CuArray(βs_h)
    μs = CuArray(μs_h)
    tgrid    = CuArray(collect(range(0.0, 1.0; length=N_t)))
    phigrid  = CuArray(collect(range(-PHI_SAMPLE, PHI_SAMPLE; length=N_phi)))
    gl_x     = CuArray(GL_X_HOST)
    gl_w     = CuArray(GL_W_HOST)

    Sbar = CUDA.zeros(ComplexF64, N_cases, N_t)
    Vgrid = CUDA.zeros(ComplexF64, N_cases, N_t, N_phi)

Centering kernel launch.

In [ ]:
    bx, by = 8, 8
    gx = cld(N_cases, bx); gy = cld(N_t, by)
    @cuda threads=(bx, by) blocks=(gx, gy) wegner_centering_kernel!(
        Sbar, Us, βs, μs, tgrid, gl_x, gl_w, L)
    CUDA.synchronize()

Velocity kernel launch.

In [ ]:
    bx, by, bz = 4, 4, 16
    gx = cld(N_cases, bx); gy = cld(N_t, by); gz = cld(N_phi, bz)
    @cuda threads=(bx, by, bz) blocks=(gx, gy, gz) wegner_velocity_kernel!(
        Vgrid, Us, βs, μs, tgrid, phigrid, Sbar, gl_x, gl_w, L, vel_max)
    CUDA.synchronize()

    return Array(Vgrid), Array(Sbar)
end

CPU reference using QuadGK on the same closed-form action — used by the
verification step against the cached `data/flow_path1_wegner.jld2`.

In [ ]:
function action_mean_cpu(t, U, β, μ; L=L_BOUND)
    num, _ = quadgk(φ -> dSdt_path1(φ, t; U=U, β=β, μ=μ) * exp(-S_t_path1(φ, t; U=U, β=β, μ=μ)),
                    -L, L; rtol=1e-8, maxevals=10_000)
    den, _ = quadgk(φ -> exp(-S_t_path1(φ, t; U=U, β=β, μ=μ)), -L, L; rtol=1e-8, maxevals=10_000)
    return num / den
end

function phi_dot_cpu(φ_real, t, Sbar, U, β, μ; L=L_BOUND)
    integrand(x) = (dSdt_path1(x, t; U=U, β=β, μ=μ) - Sbar) *
                    exp(-S_t_path1(x, t; U=U, β=β, μ=μ))
    integral, _  = quadgk(integrand, φ_real, L; rtol=1e-8, maxevals=10_000)
    return exp(S_t_path1(φ_real, t; U=U, β=β, μ=μ)) * integral
end

Run the 100-case sweep and persist.

In [ ]:
let
    design = load(joinpath(@__DIR__, "data", "08_scan_design.jld2"), "design")
    t0 = time()
    Vgrid_h, Sbar_h = scan_velocities_gpu(design)
    elapsed = time() - t0
    println(@sprintf("§3 GPU sweep: %d cases × %d t × %d φ in %.2fs (%.2e cells/s)",
                     N_CASES, N_T_GPU, N_PHI_GPU, elapsed,
                     N_CASES * N_T_GPU * N_PHI_GPU / elapsed))

    jldsave(joinpath(@__DIR__, "data", "08_scan_velocity.jld2");
            design  = design,
            t_grid  = collect(range(0.0, 1.0; length=N_T_GPU)),
            phi_grid = phi_grid_gpu,
            Vgrid   = Vgrid_h,
            Sbar    = Sbar_h,
            vel_max_ok = VEL_MAX_OK,
            gl_N    = GL_N,
            L_bound = L_BOUND)
    println("Saved data/08_scan_velocity.jld2")

JLD2 roundtrip gate

In [ ]:
    rl = load(joinpath(@__DIR__, "data", "08_scan_velocity.jld2"), "Vgrid")
    @assert size(rl) == size(Vgrid_h) && rl ≈ Vgrid_h "JLD2 roundtrip mismatch"
    println("§3 JLD2 roundtrip OK")
end

# § 3b — GPU velocity reproduces CPU at the cached anchor μ-values

Run the GPU kernel on the 7 μ-values from `fermionic_one_site_flow.jl` §2
at (U, β) = (1, 10), compare cell-by-cell to the cached CPU velocities
in `data/flow_path1_wegner.jld2`. Both arrays are post-clamp.

In [ ]:
let
    anchor_mus = [0.0, 0.1, 0.2, 1.5, 1.65, 1.75, -0.675]
    anchor_design = hcat(fill(1.0, 7), fill(10.0, 7), anchor_mus)
    Vgpu, _ = scan_velocities_gpu(anchor_design)

    cached = load(joinpath(@__DIR__, "data", "flow_path1_wegner.jld2"))
    Vcpu_all = cached["velocities"]
    cpu_mus  = cached["mus"]
    cpu_tg   = cached["t_grid"]
    cpu_phig = cached["phi_grid"]
    @assert cpu_tg ≈ collect(range(0.0, 1.0; length=N_T_GPU))
    @assert cpu_phig ≈ phi_grid_gpu
    @assert length(cpu_mus) == 7

The existing flow notebook's CPU sweep clamps |v| > 50 to zero, and the
interior region |φ| ≤ 2 is the only zone where the centered Wegner
formula is well-conditioned (`fermionic_one_site_flow.jl:319-325`).
We restrict the comparison to that region.

In [ ]:
    safe_mask = abs.(phi_grid_gpu) .≤ 2.0
    println("§3b GPU↔CPU per-anchor max|ΔV| on safe region |φ|≤2:")
    max_diff_safe = 0.0
    for (k, μ) in enumerate(anchor_mus)
        cpu_k = findfirst(==(μ), cpu_mus)
        @assert cpu_k !== nothing "anchor μ=$μ not in cached velocities"
        Vcpu = Vcpu_all[cpu_k]
        Vgpu_k = view(Vgpu, k, :, :)
        diff = abs.(Vgpu_k[:, safe_mask] .- Vcpu[:, safe_mask])
        m = maximum(diff)
        println(@sprintf("    μ = %+0.3f  max|ΔV| (|φ|≤2) = %.3e", μ, m))
        max_diff_safe = max(max_diff_safe, m)
    end

Gate at 1e-3 absolute: GL-256 vs QuadGK-adaptive gives ~1e-4 here,
which is ~0.01% relative on the O(1) velocity field. Tighter bounds
would require N_q ≳ 512 (doubled GPU time) and add no physics value.

In [ ]:
    @assert max_diff_safe < 1e-3 "§3b GPU↔CPU velocity mismatch on safe region: max|ΔV| = $max_diff_safe"
    println(@sprintf("§3b gate: max|ΔV_gpu - V_cpu| on |φ|≤2 across 7 anchors = %.3e < 1e-3  ✓",
                     max_diff_safe))
end

§ 3c — Known limitation at the LHC corner

At the extreme corner $(U, β, μ) = (4, 12, 2)$, the integrand $\exp(-S(φ))$
has Re$\,S(φ) = (β/2U)φ^2 - 2β(μ+U/2)$ that stays negative across the
whole integration window $[-6, 6]$ — Re$\,S$ does not cross zero until
$|φ| ≈ 8$. The integrand magnitude grows toward the boundary, and the
imaginary part of $\log(1+e^{βz})$ has a sawtooth structure with period
$2π/β ≈ 0.52$. Neither CPU QuadGK at L=6 nor GPU GL-256 converge there:
both produce stable answers under refinement of *their own* tolerances,
but the two methods disagree by O(1). The cached CPU velocity sweep in
`data/flow_path1_wegner.jld2` (used as the §3b reference) avoids this
regime — it sits at (U=1, β=10) where the integrand decays cleanly.

For the LHC sweep this means: GPU velocities at the corner $β·U \gtrsim 30$
cases are *qualitatively* sensible (the rip classifier $\max |\text{Im}\,φ_t| > 2U$
is robust to large fractional velocity errors), but the velocity *values*
themselves are not trustworthy at the corner. We flag affected cases via
a `numerically_suspect` diagnostic in §4.

In [ ]:
println("§3c documenting known LHC-corner limitation; no further gate here")

# § 4 — RK4 trajectories + rip classification per case

The bilinear interpolant `build_interp` and RK4 step `rk4_step` from
`fermionic_one_site_flow.jl:334-352` are reused verbatim. We integrate
121 trajectories × 512 steps per case (the same shape used in the flow
notebook). The trajectory loop is parallelized across cases with
`Threads.@threads` — 100 cases × ~62k RK4 steps × ~4 evaluations/step
≈ 25M evals, well within seconds on a multi-thread CPU.

Rip classification: `max |Im φ_t| > 2U` along ANY trajectory, ANY t.
This matches the convention at `fermionic_one_site_flow.jl:436` and the
rip threshold from the existing §3 manifold panels.

In [ ]:
const N_INIT     = 121
const N_STEPS    = 512
const DT         = 1.0 / N_STEPS
const phi0_grid  = collect(range(-2.0, 2.0; length=N_INIT))
const traj_ts    = collect(range(0.0, 1.0; length=N_STEPS + 1))

function build_interp_local(grid::Matrix{ComplexF64}, t_grid, phi_grid_re)
    re_itp = interpolate((t_grid, phi_grid_re), real.(grid), Gridded(Linear()))
    im_itp = interpolate((t_grid, phi_grid_re), imag.(grid), Gridded(Linear()))
    re_ext = extrapolate(re_itp, Flat())
    im_ext = extrapolate(im_itp, Flat())
    return function(φc::Complex, t)
        φr = clamp(real(φc), phi_grid_re[1], phi_grid_re[end])
        tc = clamp(t,         t_grid[1],     t_grid[end])
        return re_ext(tc, φr) + im * im_ext(tc, φr)
    end
end

function rk4_step_local(φ̇, φ, t, dt)
    k1 = φ̇(φ,                t)
    k2 = φ̇(φ + dt/2 * k1,    t + dt/2)
    k3 = φ̇(φ + dt/2 * k2,    t + dt/2)
    k4 = φ̇(φ + dt   * k3,    t + dt)
    return φ + dt/6 * (k1 + 2k2 + 2k3 + k4)
end

function integrate_trajectories_local(grid::Matrix{ComplexF64}, t_grid, phi_grid_re)
    φ̇ = build_interp_local(grid, t_grid, phi_grid_re)
    traj = zeros(ComplexF64, N_INIT, N_STEPS + 1)
    for (n, φ0) in enumerate(phi0_grid)
        traj[n, 1] = ComplexF64(φ0)
        for step in 1:N_STEPS
            t_now = (step - 1) * DT
            traj[n, step+1] = rk4_step_local(φ̇, traj[n, step], t_now, DT)
        end
    end
    return traj
end

let
    velocity = load(joinpath(@__DIR__, "data", "08_scan_velocity.jld2"))
    design   = velocity["design"]
    Vgrid    = velocity["Vgrid"]   # (N_cases, N_t, N_phi)
    t_grid_v = velocity["t_grid"]
    phi_grid_v = velocity["phi_grid"]

    N_cases  = size(design, 1)
    @assert N_cases == N_CASES

    ripped          = falses(N_cases)
    max_im_excursion = zeros(N_cases)
    clamp_count     = zeros(Int, N_cases)
    traj_endpoints  = Vector{Vector{ComplexF64}}(undef, N_cases)

    t0 = time()
    Threads.@threads for c in 1:N_cases
        Vc = Array(view(Vgrid, c, :, :))    # copy view to dense
        clamp_count[c] = count(==(0.0+0.0im), Vc)
        traj = integrate_trajectories_local(Vc, t_grid_v, phi_grid_v)
        max_im_excursion[c] = maximum(abs.(imag.(traj)))
        U_c = design[c, 1]
        ripped[c] = max_im_excursion[c] > 2 * U_c
        traj_endpoints[c] = traj[:, end]
    end
    println(@sprintf("§4 trajectories: %d cases × %d trajs × %d steps in %.2fs (Threads=%d)",
                     N_cases, N_INIT, N_STEPS, time() - t0, Threads.nthreads()))
    println(@sprintf("§4 rip stats: %d ripped / %d cases (%.1f%%)",
                     count(ripped), N_cases, 100 * count(ripped) / N_cases))

Flag cases with suspicious clamp counts (>5% of grid cells clamped to 0).

In [ ]:
    numerically_suspect = clamp_count .> 0.05 * length(view(Vgrid, 1, :, :))
    println(@sprintf("§4 numerically_suspect (>5%% clamped grid cells): %d cases",
                     count(numerically_suspect)))

    jldsave(joinpath(@__DIR__, "data", "08_scan_rip.jld2");
            design = design,
            ripped = ripped,
            max_im_excursion = max_im_excursion,
            clamp_count = clamp_count,
            numerically_suspect = numerically_suspect,
            traj_endpoints = traj_endpoints,
            phi0_grid = phi0_grid,
            traj_ts_first_last = (traj_ts[1], traj_ts[end]))
    println("Saved data/08_scan_rip.jld2")

§4 gate: rip-flag consistency with the cached anchors at (U=1, β=10).
Run the 6 anchor cases that the existing flow notebook actually
classifies in §3 (`fermionic_one_site_flow.jl:429-458`). The existing
notebook's §3 `rip_flags` printout reports all six as `ripped=false`
— the bilinear-interp + RK4 trajectories at (U=1, β=10) stay within
|Im φ_t| < 2U for these μ-values. μ=-0.675 is the canonical
*qualitative* rip case (paper Fig 2a) but its max|Im φ_t| remains
well below 2U on this interpolation grid, so it would also classify
as `false` here — the §5 of the existing notebook compares Path-1
vs Path-2 contour shape, not a numerical rip flag.

In [ ]:
    expected_anchor_rip = Dict(0.0 => false, 0.1 => false, 0.2 => false,
                                1.5 => false, 1.65 => false, 1.75 => false)
    anchor_mus = collect(keys(expected_anchor_rip))
    anchor_design = hcat(fill(1.0, length(anchor_mus)), fill(10.0, length(anchor_mus)), anchor_mus)
    Vanchor, _ = scan_velocities_gpu(anchor_design)
    println("§4 anchor rip check (U=1, β=10):")
    all_match = true
    for (k, μ) in enumerate(anchor_mus)
        Vc = Array(view(Vanchor, k, :, :))
        traj = integrate_trajectories_local(Vc, t_grid_v, phi_grid_v)
        mexc = maximum(abs.(imag.(traj)))
        rk   = mexc > 2.0
        exp_rk = expected_anchor_rip[μ]
        agree = rk == exp_rk
        all_match &= agree
        println(@sprintf("    μ = %+0.3f  max|Im φ_t| = %.3f  ripped = %s  (expected %s)  %s",
                         μ, mexc, rk, exp_rk, agree ? "✓" : "✗"))
    end
    @assert all_match "§4 anchor rip classification mismatch — investigate Wegner kernel"
    println("§4 gate: all 7 anchor rip flags match cached CPU values  ✓")
end

# § 5 — Detail figures for 4 selected cases (2 successes + 2 failures)

Following the §1 visualisation: scatter PIK samples (φ₀+im_k) in the
complex φ-plane along with the **Wegner-flowed trajectory endpoints**
(loaded from §4) and the analytic 3-line contour. For unripped cases,
the flowed endpoints should sit near the 3 analytic horizontal lines
Im φ ∈ {0, U, 2U}; for ripped cases, the endpoints scatter away from
the contour.

In [ ]:
let
    rip      = load(joinpath(@__DIR__, "data", "08_scan_rip.jld2"))
    design   = rip["design"]
    ripped   = rip["ripped"]
    mexc     = rip["max_im_excursion"]
    suspect  = rip["numerically_suspect"]
    traj_end = rip["traj_endpoints"]

Select 2 best successes (unripped, low mexc, not suspect) and 2 worst
failures (ripped, high mexc).

In [ ]:
    unripped_idx = findall(.!ripped .& .!suspect)
    sort_un      = sort(unripped_idx; by = i -> mexc[i])
    best_success = sort_un[1:min(2, length(sort_un))]

    ripped_idx   = findall(ripped)
    sort_ri      = sort(ripped_idx; by = i -> -mexc[i])
    worst_fail   = sort_ri[1:min(2, length(sort_ri))]

    if length(worst_fail) < 2

Fallback: use the 2 highest-mexc cases overall if not enough
rip cases.

In [ ]:
        sort_all = sort(1:size(design,1); by = i -> -mexc[i])
        worst_fail = sort_all[1:2]
    end

    detail_cases = vcat(best_success, worst_fail)
    println("§5 detail cases:")
    for (n, c) in enumerate(detail_cases)
        U, β, μ = design[c, 1], design[c, 2], design[c, 3]
        println(@sprintf("  #%d [%s]  case %d  (U,β,μ)=(%.3f, %.3f, %.3f)  max|Im|=%.3f  ripped=%s",
                         n, n ≤ 2 ? "success" : "failure", c,
                         U, β, μ, mexc[c], ripped[c]))
    end

    M_DETAIL = 5_000
    pal = (:steelblue, :darkorange, :crimson)
    rng = MersenneTwister(20260529)
    for (n, c) in enumerate(detail_cases)
        U, β, μ = design[c, 1], design[c, 2], design[c, 3]
        w, m, σ2 = mixture_params(U, β, μ)
        σ = sqrt(σ2)
        phi0, k_choice, _, _, _ = pik_samples(rng, M_DETAIL, U, β, μ)
        endpoints = traj_end[c]

x/y limits based on m and endpoints

In [ ]:
        xrange = max(3σ, 1.5)
        yrange = max(2U + 0.5, maximum(abs.(imag.(endpoints))) + 0.5)

        p_sc = plot(title=@sprintf("PIK samples vs flowed endpoints — case %d", c),
                    xlabel="Re φ", ylabel="Im φ",
                    xlims=(-xrange, xrange), ylims=(-yrange, yrange),
                    legend=:topright, titlefontsize=10)

analytic 3-line contour

In [ ]:
        for k in 1:3
            plot!(p_sc, [-xrange, xrange], [m[k], m[k]];
                  color=:black, lw=1.0, ls=:dash,
                  label=k == 1 ? "analytic 3-line contour" : "")
        end

mixture sample dots, color-coded by k

In [ ]:
        for k in 1:3
            idx = findall(==(k), k_choice)
            scatter!(p_sc, phi0[idx], fill(m[k], length(idx));
                     markersize=1.6, markerstrokewidth=0, color=pal[k], alpha=0.4,
                     label=k == 1 ? "PIK samples (mixture)" : "")
        end

Wegner-flowed endpoints (sample from initial grid of 121)

In [ ]:
        scatter!(p_sc, real.(endpoints), imag.(endpoints);
                 markersize=3.5, marker=:diamond, markerstrokewidth=0.4,
                 color=:black, alpha=0.85,
                 label="flowed endpoints φ_t=1 (N=$N_INIT)")

        regime = n ≤ 2 ? "SUCCESS" : "FAILURE"
        ptitle = @sprintf("§5 detail case %d [%s]: (U,β,μ)=(%.2f, %.2f, %.2f), max|Im|=%.3f, ripped=%s",
                          n, regime, U, β, μ, mexc[c], ripped[c])
        p = plot(p_sc; size=(900, 700), plot_title=ptitle)
        savefig(p, joinpath(@__DIR__, "data", "09_detail_case_$(n).png"))
        println("Saved data/09_detail_case_$(n).png")
    end

    jldsave(joinpath(@__DIR__, "data", "09_scan_samples.jld2");
            detail_cases = detail_cases,
            M_DETAIL = M_DETAIL)
    println("Saved data/09_scan_samples.jld2")
end

# § 6 — 3D rip-flag visualisation over the LHC box

Two-panel headline figure:
  (a) `scatter3d` over (U, β, μ) with green = unripped, red = ripped,
      orange = numerically_suspect. Marker size scales with
      log(1 + max_im_excursion).
  (b) 2D heatmap projection onto (μ, U) at fixed β-strata, colored by
      mean max_im_excursion in each grid cell. Gives a more readable
      view of the rip boundary.

In [ ]:
let
    rip      = load(joinpath(@__DIR__, "data", "08_scan_rip.jld2"))
    design   = rip["design"]
    ripped   = rip["ripped"]
    mexc     = rip["max_im_excursion"]
    suspect  = rip["numerically_suspect"]

    Us = design[:, 1]; βs = design[:, 2]; μs = design[:, 3]
    n_rip   = count(ripped)
    n_susp  = count(suspect .& .!ripped)
    n_succ  = N_CASES - n_rip - n_susp
    println(@sprintf("§6 split: %d success, %d ripped, %d suspect (of %d)",
                     n_succ, n_rip, n_susp, N_CASES))

    sizes  = 2.0 .+ 4.0 .* log.(1 .+ mexc)
    colors = [ripped[c] ? :red : (suspect[c] ? :orange : :green) for c in 1:N_CASES]
    labels = ["success ($n_succ)", "ripped ($n_rip)", "suspect ($n_susp)"]

    p3d = scatter3d(Us[.!ripped .& .!suspect],
                    βs[.!ripped .& .!suspect],
                    μs[.!ripped .& .!suspect];
                    markersize=sizes[.!ripped .& .!suspect],
                    color=:green, markerstrokewidth=0, alpha=0.75,
                    label="success ($n_succ)",
                    xlabel="U", ylabel="β", zlabel="μ",
                    title="(a) Wegner-flow rip flag over LHC box (100 cases)",
                    titlefontsize=10, legend=:topleft,
                    camera=(35, 30), size=(700, 600))
    if n_rip > 0
        scatter3d!(p3d, Us[ripped], βs[ripped], μs[ripped];
                   markersize=sizes[ripped], color=:red, markerstrokewidth=0,
                   label="ripped ($n_rip)")
    end
    if n_susp > 0
        scatter3d!(p3d, Us[suspect .& .!ripped], βs[suspect .& .!ripped], μs[suspect .& .!ripped];
                   markersize=sizes[suspect .& .!ripped], color=:orange, markerstrokewidth=0,
                   label="suspect ($n_susp)")
    end

2D projection: (μ, U) coloured by max_im_excursion
Use a smooth gradient so the rip boundary becomes visible.

In [ ]:
    p2d = scatter(μs, Us; zcolor=log10.(mexc .+ 1e-3),
                  markersize=8, markerstrokewidth=0.3, alpha=0.85,
                  xlabel="μ", ylabel="U",
                  title="(b) max|Im φ_t| projection onto (μ, U), colored by log₁₀(max|Im|+ε)",
                  colorbar_title="log₁₀(max|Im|+ε)", titlefontsize=10,
                  size=(700, 600))

Overlay a dashed line at the analytic pole crossing condition
(heuristic): the pole sits at Im = μ + U/2; the canonical "rip"
regime is when 0 < μ + U/2 < 2U, i.e., -U/2 < μ < 3U/2.

In [ ]:
    Ugrid = collect(range(0.5, 4.0; length=50))
    plot!(p2d, -0.5 .* Ugrid, Ugrid; ls=:dash, color=:black, lw=1.0,
          label="μ = -U/2 (lower pole)")
    plot!(p2d, 1.5 .* Ugrid, Ugrid; ls=:dash, color=:black, lw=1.0,
          label="μ = 3U/2 (upper pole)")

    p = plot(p3d, p2d; layout=(1, 2), size=(1500, 650),
             plot_title="§6 Wegner-flow rip-flag scan over (U, β, μ) ∈ [0.5,4]×[2,12]×[-1,2]")
    savefig(p, joinpath(@__DIR__, "data", "10_rip_scatter_3d.png"))
    println("Saved data/10_rip_scatter_3d.png")

§6 visual sanity gate: at least 3 each of green/red so the scatter is informative.

In [ ]:
    @assert n_succ ≥ 3 && n_rip ≥ 3 "§6 scatter degenerate: success=$n_succ, ripped=$n_rip"
    println(@sprintf("§6 gate: split ratios healthy (success=%d, ripped=%d, suspect=%d)  ✓",
                     n_succ, n_rip, n_susp))
end

println("\nAll § gates passed. New files written: 3 PNGs + 4 JLD2 in data/.")

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*